# Finding/opening ACCESS-OM3 `MC_25km_jra_ryf-1.0-beta` test run output using intake

This notebook demonstrates how to using intake-esm to find and load data data from the ACCESS-OM3 `MC_25km_jra_ryf-1.0-beta` test run.

For more information about using intake-esm to find and load data, see:
- the [intake-esm documentation](https://intake-esm.readthedocs.io/en/stable/)
- [this section](https://access-nri-intake-catalog.readthedocs.io/en/latest/usage/quickstart.html#using-an-intake-esm-datastore) of the access-nri-intake-catalog documentation

In [ ]:
#This cell must be in all notebooks!
#It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from attenuation_models import *
test()

### USER EDIT start
# esm_file = "/g/data/ps29/nd0349/runs/access-om3/archive/mom6-cice6_ryf_rel/intake_esm_ds.json"
# esm_file = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_ryf_nomixing/experiment_datastore.json"
# esm_file = "/g/data/ol01/access-om3-output/access-om3-025/MC_25km_jra_ryf-1.0-beta-cdfb3543/experiment_datastore.json"
# esm_file = '/g/data/ol01/access-om3-output/access-om3-025/25km-iaf-test-for-AK-expt-7df5ef4c/datastore.json'
esm_file = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_ryf/experiment_datastore.json"
esm_file2 = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_ryf_defaultmixing/experiment_datastore.json"
esm_file3 = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_ryf_nomixing/experiment_datastore.json"
esm_file4 = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_ryf_oldmixing/experiment_datastore.json"


# esm_file2 = "/scratch/ps29/nd0349/access-om3/archive/IC4M2-MCW-100km_jra_ryf/experiment_datastore.json"
dpi=300
### USER EDIT stop

import os
from matplotlib import rcParams
%matplotlib inline
rcParams["figure.dpi"]= dpi

plotfolder=f"/g/data/{os.environ['PROJECT']}/{os.environ['USER']}/access-om3-figs/"
os.makedirs(plotfolder, exist_ok=True)

 # a similar cell under this means it's being run in batch
print("ESM datastore path: ",esm_file)
print("Plot folder path: ",plotfolder)

In [ ]:
import xarray as xr
import cf_xarray
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client

In [ ]:
client = Client(threads_per_worker=1)
client

In [ ]:
print(client.dashboard_link)

### Open the intake-esm datastore

In [ ]:
intake.open_esm_datastore(esm_file)

In [ ]:
COLUMNS_WITH_ITERABLES = [
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
]

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)


In [ ]:
# esmcat = json.load(f)

In [ ]:
datastore2 = intake.open_esm_datastore(
    esm_file2,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)


### What ocean variables are available at monthly frequency?

In [ ]:
def available_variables(datastore):
    """Return a pandas dataframe summarising the variables in a datastore"""
    variable_columns = [col for col in datastore.df.columns if "variable" in col]
    return (
        datastore.df[variable_columns]
        .explode(variable_columns)
        .drop_duplicates()
        .set_index("variable")
        .sort_index()
    )

In [ ]:
datastore_filtered = datastore.search(realm="ocean", frequency="1mon")

available_variables(datastore_filtered)

### Import dataset 1

In [ ]:
ds1 = datastore.search(variable="mlotst", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                         ).to_dask().compute()
coords = coords.fillna(0.0)

coords = coords.rename({
    "geolon": "lon",
    "geolat": "lat"
})
ds1 = ds1.assign_coords(coords)
# Make CF compliant
ds1['lon'].attrs['standard_name'] = 'longitude'
ds1['lon'].attrs['units'] = 'degrees_east'
ds1['lat'].attrs['standard_name'] = 'latitude'
ds1['lat'].attrs['units'] = 'degrees_north'
ds1 = ds1.set_coords(['lon', 'lat'])

ds1


In [ ]:
ds1_cice = datastore.search(variable="aice_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"nj": -1, "ni": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore.search(variable=["TLAT", "TLON"], realm="seaIce", frequency="fx",
                          ).to_dask().compute()

coords = coords.fillna(0.0)
# ds2_cice = xr.merge(ds2_cice, coords)
ds1_cice['TLAT'] = coords['TLAT']
ds1_cice = ds1_cice.set_coords(['TLON', 'TLAT'])
ds1_cice

### Import dataset 2

In [ ]:
ds2 = datastore2.search(variable="mlotst", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore2.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                          ).to_dask().compute()
coords = coords.rename({
    "geolon": "lon",
    "geolat": "lat"
})
coords = coords.fillna(0.0)
ds2 = ds2.assign_coords(coords)

ds2['lon'].attrs['standard_name'] = 'longitude'
ds2['lon'].attrs['units'] = 'degrees_east'
ds2['lat'].attrs['standard_name'] = 'latitude'
ds2['lat'].attrs['units'] = 'degrees_north'
ds2 = ds2.set_coords(['lon', 'lat'])
ds2

In [ ]:
ds2_cice = datastore2.search(variable="aice_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"ni": -1, "nj": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

coords = datastore2.search(variable=["TLAT", "TLON"], realm="seaIce", frequency="fx",
                          ).to_dask().compute()
coords = coords.fillna(0.0)
# ds2_cice = xr.merge(ds2_cice, coords)
ds2_cice['TLAT'] = coords['TLAT']
ds2_cice = ds2_cice.set_coords(['TLON', 'TLAT'])
ds2_cice

### Import dataset 3

In [ ]:
datastore3 = intake.open_esm_datastore(
    esm_file3,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)
datastore3.unique().path[0]

In [ ]:
ds3 = datastore3.search(variable="mlotst", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore3.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                          ).to_dask().compute()
coords = coords.rename({
    "geolon": "lon",
    "geolat": "lat"
})
coords = coords.fillna(0.0)
ds3 = ds3.assign_coords(coords)

ds3['lon'].attrs['standard_name'] = 'longitude'
ds3['lon'].attrs['units'] = 'degrees_east'
ds3['lat'].attrs['standard_name'] = 'latitude'
ds3['lat'].attrs['units'] = 'degrees_north'
ds3 = ds3.set_coords(['lon', 'lat'])
ds3

In [ ]:
ds3_cice = datastore3.search(variable="aice_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"ni": -1, "nj": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

coords = datastore3.search(variable=["TLAT", "TLON"], realm="seaIce", frequency="fx",
                          ).to_dask().compute()
coords = coords.fillna(0.0)
ds3_cice['TLAT'] = coords['TLAT']
ds3_cice = ds3_cice.set_coords(['TLON', 'TLAT'])
ds3_cice

### Import dataset 4

In [ ]:
datastore4 = intake.open_esm_datastore(
    esm_file4,
    columns_with_iterables=COLUMNS_WITH_ITERABLES
)
datastore4.unique().path[0]

In [ ]:
ds4 = datastore4.search(variable="mlotst", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore4.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                          ).to_dask().compute()
coords = coords.rename({
    "geolon": "lon",
    "geolat": "lat"
})
coords = coords.fillna(0.0)
ds4 = ds4.assign_coords(coords)

ds4['lon'].attrs['standard_name'] = 'longitude'
ds4['lon'].attrs['units'] = 'degrees_east'
ds4['lat'].attrs['standard_name'] = 'latitude'
ds4['lat'].attrs['units'] = 'degrees_north'
ds4 = ds4.set_coords(['lon', 'lat'])
ds4

In [ ]:
ds4_cice = datastore4.search(variable="aice_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"ni": -1, "nj": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

coords = datastore4.search(variable=["TLAT", "TLON"], realm="seaIce", frequency="fx",
                          ).to_dask().compute()
coords = coords.fillna(0.0)
ds4_cice['TLAT'] = coords['TLAT']
ds4_cice = ds4_cice.set_coords(['TLON', 'TLAT'])
ds4_cice

### Regrid onto 100km

In [ ]:
# print(ds_src.lon.shape, ds_src.lat.shape)
# print(ds_tgt.lon.shape, ds_tgt.lat.shape)

# for v in ds_src.data_vars:
#     print(v, ds_src[v].shape)

# for v in ds_tgt.data_vars:
#     print(v, ds_tgt[v].shape)

In [ ]:
# 

In [ ]:
import xarray as xr
import xesmf as xe

# --- Example: assume you already have two datasets ---
ds_src = ds1.isel(time=slice(0,12))
ds_tgt = ds2

ds_src = ds_src.rename({"yh": "y", "xh": "x"})
ds_tgt = ds_tgt.rename({"yh": "y", "xh": "x"})
ds_src = ds_src.set_coords(["lon", "lat"])
ds_tgt = ds_tgt.set_coords(["lon", "lat"])

grid_src = {"lon": ds_src.lon, "lat": ds_src.lat}
grid_tgt = {"lon": ds_tgt.lon, "lat": ds_tgt.lat}

# xESMF expects grids to have lat/lon names.
# If your coordinates have different names, rename:
# ds_src = ds_src.rename({'yt_ocean': 'lat', 'xt_ocean': 'lon'})
# ds_tgt = ds_tgt.rename({'yh': 'lat', 'xh': 'lon'})

# --- Build regridder ---
regridder = xe.Regridder(
    grid_src,
    grid_tgt,
    method="bilinear",
    periodic=False,
    ignore_degenerate=True,   # required for MOM6 / tripolar grids
    reuse_weights=False,
)

# --- Apply to data variable(s) ---
# If your dataset has multiple variables, select one or apply to all.
mlotst_regridded = regridder(ds_src["mlotst"])

# --- Save or use the result ---
# ds_regridded.to_netcdf('regridded.nc')

mlotst_regridded

### Load and plot obs data from DeBoyer Montegut (2023)
https://doi.org/10.17882/91774

In [ ]:
MLDobs = xr.open_dataset('/g/data/av17/access-nri/OM3/MLD-DeBoyerMontegut2023/mld_dr003_ref10m_v2023.nc')['mld_dr003']
MLDobs.attrs['units'] = MLDobs.attrs['unit']  # fix so plot works

# TODO: append copy of westernmost data to eastern end to avoid gap in plot

In [ ]:
# small BUG: mean of monthly means is not mean of days in that month (eg Feb gets slightly more heavily weighted)
# MLDobs_JFM_mean = MLDobs.isel(time=[0, 1, 2]).mean('time').load()
# MLDobs_JAS_mean = MLDobs.isel(time=[6, 7, 8]).mean('time').load()
MLDobs

## Plot the comparison in MLD

In [ ]:
from pathlib import Path

exp1 = Path(esm_file).parent.name
exp2 = Path(esm_file2).parent.name

print(exp1)  # mom6-cice6_ryf_rel
print(exp2)  # IC4M8-MCW-100km_jra_ryf

In [ ]:
ds2_cice.attrs['intake_esm_attrs:realm']

### Plot against Obs

In [ ]:
time_idx = -1
hemisphere = "south"
dims = [1, 3]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

# settings = get_plot_settings("mlotst")

# plot(MLDobs_JAS_mean,
#     levels=51,
#     vmin=0,
#     vmax=500,
#     extend="max",
#     cmap='viridis',
#     title=f"Observed mixed layer depth JAS mean (DeBoyer Montegut, 2023)"
#     )

MLDobs.isel(time=time_idx).plot(ax=axes[0], #x="geolon", y="geolat", 
                    vmin=0,
                    vmax=500,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
                    cbar_kwargs={'label': 'Mixed Layer Depth Obs [m]'}
) 

ds1["mlotst"].isel(time=time_idx).plot(ax=axes[1], x="lon", y="lat", 
                    vmin=0,
                    vmax=500,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[1], ds1_cice.isel(time=time_idx), hemisphere, projection)

ds2["mlotst"].isel(time=time_idx).plot(ax=axes[2], x="lon", y="lat", 
                    vmin=0,
                    vmax=500,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[2], ds2_cice.isel(time=time_idx), hemisphere, projection)


# diff = ds2["mlotst"].isel(time=time_idx) - ds1["mlotst"].isel(time=time_idx)
# diff = diff.assign_coords(coords)
# diff.plot(ax=axes[2], x="geolon", y="geolat", 
#                     vmin=-abs(diff).max().values,
#                     vmax=abs(diff).max().values,
#                     cmap=cmo.balance,
#                     transform=ccrs.PlateCarree(),
#                     cbar_kwargs={"label": "Difference [m]"} 
# ) 
fig.suptitle(f"Comparison with observations of {exp1} and {exp2}", fontsize=16)

### Plot comparison

In [ ]:
ds1_cice.attrs['intake_esm_attrs:realm']

In [ ]:
time_idx = 0
hemisphere = "south"
dims = [1, 3]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

settings = get_plot_settings("ICE")

ds1["mlotst"].isel(time=time_idx).plot(ax=axes[0], x="lon", y="lat", 
                    vmin=0,
                    vmax=500,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 

ds3["mlotst"].isel(time=time_idx).plot(ax=axes[1], x="lon", y="lat", 
                    vmin=0,
                    vmax=500,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 

diff = ds3["mlotst"].isel(time=time_idx) - ds1["mlotst"].isel(time=time_idx)
diff = diff.assign_coords(coords)
diff.plot(ax=axes[2], x="lon", y="lat", 
                    vmin=-10,#-abs(diff).max().values,
                    vmax=10,#abs(diff).max().values,
                    cmap=cmo.balance,
                    transform=ccrs.PlateCarree(),
                    cbar_kwargs={"label": "Difference [m]"} 
) 
# fig.suptitle(rf"Comparison of {exp1} and {exp2}", fontsize=16)

In [ ]:
datastore.search(variable="aice_m", frequency="1mon")

### Plot all runs

In [ ]:
time_idx = 6
hemisphere = "south"
dims = [1, 5]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

MLDobs.isel(time=time_idx).plot(ax=axes[0], #x="geolon", y="geolat", 
                    vmin=0,
                    vmax=600,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
                    cbar_kwargs={'label': 'Mixed Layer Depth Obs [m]'}
) 
axes[0].set_title("Observations")

ds1["mlotst"].isel(time=time_idx).plot(ax=axes[1], x="lon", y="lat", 
                    vmin=0,
                    vmax=600,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[1], ds1_cice.isel(time=time_idx), hemisphere, projection)
axes[1].set_title("MCW_100km_jra_ryf\n Uncoupled LT")


ds2["mlotst"].isel(time=time_idx).plot(ax=axes[2], x="lon", y="lat", 
                    vmin=0,
                    vmax=600,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[2], ds2_cice.isel(time=time_idx), hemisphere, projection)
axes[2].set_title("MCW_100km_jra_ryf\n Coupled LT (MOM6 defaults)")

ds3["mlotst"].isel(time=time_idx).plot(ax=axes[3], x="lon", y="lat", 
                    vmin=0,
                    vmax=600,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[3], ds3_cice.isel(time=time_idx), hemisphere, projection)
axes[3].set_title("MCW_100km_jra_ryf\n Coupled LT (no mixing)")


ds4["mlotst"].isel(time=time_idx).plot(ax=axes[4], x="lon", y="lat", 
                    vmin=0,
                    vmax=600,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 
add_ice_contours(axes[4], ds4_cice.isel(time=time_idx), hemisphere, projection)
axes[4].set_title("MCW_100km_jra_ryf\n Coupled LT (previous values)")

# diff = ds2["mlotst"].isel(time=time_idx) - ds1["mlotst"].isel(time=time_idx)
# diff = diff.assign_coords(coords)
# diff.plot(ax=axes[2], x="geolon", y="geolat", 
#                     vmin=-abs(diff).max().values,
#                     vmax=abs(diff).max().values,
#                     cmap=cmo.balance,
#                     transform=ccrs.PlateCarree(),
#                     cbar_kwargs={"label": "Difference [m]"} 
# ) 
# fig.suptitle(f"Comparison with observations of {exp1} and {exp2}", fontsize=16)

In [ ]:

ds4["mlotst"].isel(time=time_idx).time

## Check sea ice conc

In [ ]:
time_idx = 6
hemisphere = "south"
dims = [1, 3]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

settings = get_plot_settings("ICE")

ds1["aice_m"].isel(time=time_idx).plot(ax=axes[0], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=1,
                    cmap=cmo.ice,
                    transform=ccrs.PlateCarree(),
) 

ds2["aice_m"].isel(time=time_idx).plot(ax=axes[1], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=1,
                    cmap=cmo.ice,
                    transform=ccrs.PlateCarree(),
) 

# diff = ds2["aice_m"].isel(time=time_idx) - ds1["aice_m"].isel(time=time_idx)
# diff = diff.assign_coords(coords)
# diff.plot(ax=axes[2], x="geolon", y="geolat", 
#                     vmin=-abs(diff).max().values,
#                     vmax=abs(diff).max().values,
#                     cmap=cmo.balance,
#                     transform=ccrs.PlateCarree(),
#                     cbar_kwargs={"label": "Difference [m]"} 
# ) 
fig.suptitle(rf"Comparison of {exp1} and {exp2}", fontsize=16)

In [ ]:
time_idx = 0
hemisphere = "south"
dims = [1, 3]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

settings = get_plot_settings("ICE")

ds1["aice_m"].isel(time=time_idx).plot(ax=axes[0], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=1,
                    cmap=cmo.ice,
                    transform=ccrs.PlateCarree(),
) 

ds2["aice_m"].isel(time=time_idx).plot(ax=axes[1], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=1,
                    cmap=cmo.ice,
                    transform=ccrs.PlateCarree(),
) 

diff = ds2["aice_m"].isel(time=time_idx) - ds1["aice_m"].isel(time=time_idx)
diff = diff.assign_coords(coords)
diff.plot(ax=axes[2], x="geolon", y="geolat", 
                    vmin=-abs(diff).max().values,
                    vmax=abs(diff).max().values,
                    cmap=cmo.balance,
                    transform=ccrs.PlateCarree(),
                    cbar_kwargs={"label": "Difference [m]"} 
) 
fig.suptitle(rf"Comparison of {exp1} and {exp2}", fontsize=16)

In [ ]:
## And Hi

In [ ]:
ds1 = datastore.search(variable="hi_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"nj": -1, "ni": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                         ).to_dask().compute()
coords = coords.fillna(0.0)
coords = coords.rename({'xh': 'ni', 'yh': 'nj'})
ds1 = ds1.assign_coords(coords)
ds1

In [ ]:
ds2 = datastore2.search(variable="hi_m", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"ni": -1, "nj": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)
coords = datastore2.search(variable=["geolat", "geolon"], file_id='access_om3_mom6_static'
                          ).to_dask().compute()
coords = coords.fillna(0.0)
coords = coords.rename({'xh': 'ni', 'yh': 'nj'})
ds2 = ds2.assign_coords(coords)
ds2

In [ ]:
time_idx = 0
hemisphere = "south"
dims = [1, 3]
fig, axes, projection = basic_axis(dims, hemisphere=hemisphere)

settings = get_plot_settings("ICE")

ds1["hi_m"].isel(time=time_idx).plot(ax=axes[0], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=2,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 

ds2["hi_m"].isel(time=time_idx).plot(ax=axes[1], x="geolon", y="geolat", 
                    vmin=0,
                    vmax=2,
                    cmap='viridis',
                    transform=ccrs.PlateCarree(),
) 

diff = ds2["hi_m"].isel(time=time_idx) - ds1["hi_m"].isel(time=time_idx)
diff = diff.assign_coords(coords)
diff.plot(ax=axes[2], x="geolon", y="geolat", 
                    vmin=-0.5, #abs(diff).max().values,
                    vmax=0.5, #abs(diff).max().values,
                    cmap=cmo.balance,
                    transform=ccrs.PlateCarree(),
                    cbar_kwargs={"label": "Difference [m]"} 
) 
fig.suptitle(rf"Comparison of {exp1} and {exp2}", fontsize=16)

### Load monthly sea surface height (`zos`) and plot the field at the last available time

In [ ]:
zos = datastore.search(variable="zos", frequency="1mon").to_dask(
    xarray_combine_by_coords_kwargs = dict( # These kwargs can make things faster
        compat="override",
        data_vars="minimal",
        coords="minimal",
    ),
    xarray_open_kwargs = dict(
        chunks={"yh": -1, "xh": -1}, # Good for spatial operations, but not temporal
        decode_timedelta=True
    )
)

In [ ]:
proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(15,6), subplot_kw=dict(projection=proj))

zos["zos"].isel(time=-1).plot(ax=ax)
ax.coastlines()
_ = ax.gridlines()

#### Fixing the plot axes

Notice that the white land-masked regions are distorted away from the coastlines in the Arctic in the above plot. This is because a tripolar grid is used, so the grid lines are not zonal and meridional north of 65N, and consequently the nominal 1D coordinates `xh` and `yh` are incorrect. To fix this we need to use 2d coordinates `geolon` and `geolat`.

We can get these coorindates from the intake-esm datastore. However, note that `geolon` and `geolat` contain NaNs in regions where processors were masked over land. Below we replace these NaNs with zeros so that the coordinates can be used for plotting.

In [ ]:
coords = datastore.search(variable=["geolat", "geolon"]).to_dask().compute()
coords = coords.fillna(0.0)

zos = zos.assign_coords(coords)

In [ ]:
proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(15,6), subplot_kw=dict(projection=proj))

zos["zos"].isel(time=-1).plot(ax=ax, x="geolon", y="geolat")

ax.coastlines()
_ = ax.gridlines()
plt.savefig(plotfolder+'exampleout.png')

In [ ]:
client.close()